## Program 1C

Reimplementation of program 1A, but with spatial hashing to reduce the number of intersection tests. Although, like 1A this approach will eventually hit a RAM limit and crash, the number of tests is reduced by >99% and, for benchmarking, this limit was not reached on Machine 1. As with program 1B, lap steps have been consolidated.

In [1]:
n_rows = 1

In [2]:
import pandas as pd
import numpy as np
import time
from time import perf_counter_ns
from collections import defaultdict

h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

file_path = r"C:\Users\Smith\OneDrive\MSc Project\05 Final Code\Bunny Head Raw Lines for Computation Testing.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name=0,
    usecols="A:F",
    nrows=n_rows,
    header=None,
    engine="openpyxl"
)

df.columns = ["x", "y", "z", "i", "j", "k"]
A = df.to_numpy(dtype=float)

In [3]:
def point_in_CTC_spatial_hash(A, h_1, r_1, h_2, r_2, eps=1e-12):

    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    # Positive integer check for S1 theory
    if h_1 <= 0 or h_2 <= 0 or r_1 <= 0 or r_2 <= 0:
        raise ValueError("h1, h2, r1 and r2 must be positive values.")

    # Radius comparison check
    if r_2 < r_1:
        raise ValueError("r2 < r1.")

    A = np.asarray(A, dtype=float)
    lap("Input to numpy array")

    P = np.ascontiguousarray(A[:, 0:3])   # xyz points
    U = np.ascontiguousarray(A[:, 3:6])   # orientation vectors
    N = A.shape[0]
    lap("Separate Points and Vectors")

    H = h_1 + h_2

    k_sphere = max(
        (r_2*r_2 + h_1*h_1) / (2.0*h_1),
        (r_2*r_2 + H*H) / (2.0*H)
    )

    R = k_sphere
    R_sq = (R + eps) * (R + eps)

    cell_size = R
    inv_cell_size = 1.0 / cell_size

    lap("Create S1")

    grid = defaultdict(list)
    
    hit_chunks = []

    broad_cell_candidate_count = 0
    sphere_candidate_count = 0
    exact_test_count = 0

    def cell_key(point):
        return tuple(np.floor(point * inv_cell_size).astype(np.int64))

    # The Big Loop
    for i in range(N):

        Pi = P[i]
        Ui = U[i]

        # Sphere centre for i
        sphere_centre = Pi + k_sphere * Ui

        # Find all grid cells overlapping the sphere's AABB
        cmin = np.floor((sphere_centre - R) * inv_cell_size).astype(np.int64)
        cmax = np.floor((sphere_centre + R) * inv_cell_size).astype(np.int64)

        candidate_ids = []

        for cx in range(cmin[0], cmax[0] + 1):
            for cy in range(cmin[1], cmax[1] + 1):
                for cz in range(cmin[2], cmax[2] + 1):

                    ids = grid.get((cx, cy, cz))

                    if ids:
                        candidate_ids.extend(ids)

        broad_cell_candidate_count += len(candidate_ids)

        if candidate_ids:

            J = np.asarray(candidate_ids, dtype=np.int64)

            # Exact sphere membership after cell lookup
            # Returns points from cells overlapping the sphere AABB
            # Then culls the ones sitting outside the sphere, not the other way around
            W = P[J] - sphere_centre
            sphere_ok = np.einsum("ij,ij->i", W, W) <= R_sq

            J = J[sphere_ok]
            sphere_candidate_count += J.size

            if J.size:

                # Original intersection test, modified for J
                V = P[J] - Pi
                t = V @ Ui
                v2 = np.einsum("ij,ij->i", V, V)
                d_perp_sq = v2 - t*t
                d_perp_sq = np.maximum(d_perp_sq, 0.0)

                exact_test_count += J.size

                # Axial slab test
                axial_ok = (t >= -eps) & (t <= H + eps)

                # Radial test
                cone_region = t <= h_1 + eps
                cylinder_region = t > h_1 + eps

                cone_ok = (h_1*h_1*d_perp_sq) <= (r_1*r_1*t*t + eps)
                cylinder_ok = d_perp_sq <= (r_2*r_2 + eps)

                radial_ok = (cone_region & cone_ok) | (cylinder_region & cylinder_ok)

                ok = axial_ok & radial_ok

                if np.any(ok):
                    rows = np.full(np.count_nonzero(ok), i, dtype=np.int64)
                    hit_chunks.append(np.column_stack((rows, J[ok])))

        # Place current point after intersection test - mask therefore no longer needed
        grid[cell_key(Pi)].append(i)

    # Lap will just have to combine these, no idea how to separate them within the loops
    lap("Spatial hash broad phase and exact tests")

    if hit_chunks:
        hit_pairs = np.vstack(hit_chunks)
    else:
        hit_pairs = np.empty((0, 2), dtype=np.int64)

    lap("Hit List")

    print("-" * 75)
    print(f"Sphere centre offset k / radius R: {k_sphere:.6g}")
    print(f"Cell size:                       {cell_size:.6g}")
    print(f"Broad cell candidates:           {broad_cell_candidate_count:,}")
    print(f"Sphere candidates:               {sphere_candidate_count:,}")
    print(f"Intersection tests:              {exact_test_count:,}")
    print(f"Hit pairs:                       {hit_pairs.shape[0]:,}")
    print(f"{'TOTAL':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")

    return hit_pairs

In [4]:
start = time.perf_counter()
hit_pairs = point_in_CTC_spatial_hash(A, h_1, r_1, h_2, r_2)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"Function time: {elapsed_ms:.3f} ms")
print(hit_pairs.shape)

Input to numpy array                          /     0.015/ ms   total:      0.015 ms
Separate Points and Vectors                   /     0.641/ ms   total:      0.656 ms
Create S1                                     /     0.316/ ms   total:      0.972 ms
Spatial hash broad phase and exact tests      /     1.080/ ms   total:      2.052 ms
Hit List                                      /     0.130/ ms   total:      2.182 ms
---------------------------------------------------------------------------
Sphere centre offset k / radius R: 21.6667
Cell size:                       21.6667
Broad cell candidates:           0
Sphere candidates:               0
Intersection tests:              0
Hit pairs:                       0
TOTAL                                              2.419 ms
Function time: 2.822 ms
(0, 2)
